In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
# --- 1) Daten laden ---

drl_data = pd.read_csv(
    "data/data_2019-01-01_2024-01-01_hourly.csv",
    parse_dates=["time"],
    index_col="time",
)
drl_data.index = pd.to_datetime(drl_data.index, utc=True)

In [3]:
ri_profit1 = pd.read_csv("output/single_market/rolling_intrinsic/ri_basic/qh/2019/bs15cr1rto0.86mc365mt10/profit.csv")
ri_profit2 = pd.read_csv("output/single_market/rolling_intrinsic/ri_basic/qh/2023/bs15cr1rto0.86mc365mt10/profit.csv")

ri_profit = pd.concat([ri_profit1, ri_profit2], ignore_index=True)
ri_profit = ri_profit.set_index("day")  # aktuell: dtype=object mit +01:00 im String


In [4]:
# 1) Index-Strings auf 'YYYY-MM-DD HH:MM:SS' kürzen (Offset abschneiden)
idx = ri_profit.index.astype(str).str.slice(0, 19)
# z.B. '2019-01-01 00:00:00+01:00' -> '2019-01-01 00:00:00'

# 2) In echte Datumswerte umwandeln (noch ohne Zeitzone)
idx = pd.to_datetime(idx)  # dtype: datetime64[ns], immer 00:00:00

# 3) Zeitzone als UTC hinzufügen
idx = idx.tz_localize("UTC")  # dtype: datetime64[ns, UTC]

# 4) Zurück als Index setzen
ri_profit.index = idx

# Optional: Index-Name an den anderen DF anpassen
ri_profit.index.name = "date"

In [5]:
ri_profit.index

DatetimeIndex(['2019-01-01 00:00:00+00:00', '2019-01-02 00:00:00+00:00',
               '2019-01-03 00:00:00+00:00', '2019-01-04 00:00:00+00:00',
               '2019-01-05 00:00:00+00:00', '2019-01-06 00:00:00+00:00',
               '2019-01-07 00:00:00+00:00', '2019-01-08 00:00:00+00:00',
               '2019-01-09 00:00:00+00:00', '2019-01-10 00:00:00+00:00',
               ...
               '2023-12-22 00:00:00+00:00', '2023-12-23 00:00:00+00:00',
               '2023-12-24 00:00:00+00:00', '2023-12-25 00:00:00+00:00',
               '2023-12-26 00:00:00+00:00', '2023-12-27 00:00:00+00:00',
               '2023-12-28 00:00:00+00:00', '2023-12-29 00:00:00+00:00',
               '2023-12-30 00:00:00+00:00', '2023-12-31 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='date', length=1826, freq=None)

In [6]:
drl_data.drop(columns=["exaa_15min_de_lu_eur_per_mwh", "id_full_qh"], inplace=True)

In [7]:
# --- 2) Residual Load berechnen ---

drl_data["residual_load"] = (
    drl_data["load_forecast_d_minus_1_1000_total_de_lu_mw"]
    - drl_data["pv_forecast_d_minus_1_1000_de_lu_mw"]
    - drl_data["wind_offshore_forecast_d_minus_1_1000_de_lu_mw"]
    - drl_data["wind_onshore_forecast_d_minus_1_1000_de_lu_mw"]
)

In [8]:
# --- 3) Daily Aggregation über Resample ---

agg_features = [
    "epex_spot_60min_de_lu_eur_per_mwh",
    #"exaa_15min_de_lu_eur_per_mwh",
    "load_forecast_d_minus_1_1000_total_de_lu_mw",
    "pv_forecast_d_minus_1_1000_de_lu_mw",
    "wind_offshore_forecast_d_minus_1_1000_de_lu_mw",
    "wind_onshore_forecast_d_minus_1_1000_de_lu_mw",
    "residual_load",
    "id_full_h",
   # "id_full_qh",
]

agg_dict = {col: ["mean", "std", "min", "max"] for col in agg_features}

daily_agg = (
    drl_data
    .resample("D")      # täglich auf Index 'time'
    .agg(agg_dict)
)

daily_agg.columns = [
    f"{col}_{stat}" for col, stat in daily_agg.columns.to_flat_index()
]
daily_agg.index.name = "date"

daily_agg["epex_spread"] = (
    daily_agg["epex_spot_60min_de_lu_eur_per_mwh_max"]
    - daily_agg["epex_spot_60min_de_lu_eur_per_mwh_min"]
)

daily_agg["residual_load_spread"] = (
    daily_agg["residual_load_max"] - daily_agg["residual_load_min"]
)

daily_agg["day_of_week"] = daily_agg.index.dayofweek
daily_agg["month"] = daily_agg.index.month
daily_agg["year"] = daily_agg.index.year



In [9]:

df_daily = daily_agg.join(ri_profit[["profit", "cycles"]], how="inner")
print(df_daily.shape)

df_daily

(1826, 35)


,epex_spot_60min_de_lu_eur_per_mwh_mean,epex_spot_60min_de_lu_eur_per_mwh_std,epex_spot_60min_de_lu_eur_per_mwh_min,epex_spot_60min_de_lu_eur_per_mwh_max,load_forecast_d_minus_1_1000_total_de_lu_mw_mean,load_forecast_d_minus_1_1000_total_de_lu_mw_std,load_forecast_d_minus_1_1000_total_de_lu_mw_min,load_forecast_d_minus_1_1000_total_de_lu_mw_max,pv_forecast_d_minus_1_1000_de_lu_mw_mean,pv_forecast_d_minus_1_1000_de_lu_mw_std,...,id_full_h_std,id_full_h_min,id_full_h_max,epex_spread,residual_load_spread,day_of_week,month,year,profit,cycles
date,,,,,,,,,,,,,,,,,,,,,
2019-01-01 00:00:00+00:00,-6.875833,10.786639,-33.57,10.07,47744.070208,5133.243773,40308.2500,55255.2200,413.402396,742.514510,...,7.614956,-11.620194,20.045181,43.64,11515.5975,1,1,2019,68.233139,1.0
2019-01-02 00:00:00+00:00,29.104167,40.083914,-48.93,62.11,55944.564583,8385.547397,40825.7450,64053.6700,1191.025417,2161.769818,...,29.772235,-38.497470,50.482814,111.04,40389.4275,2,1,2019,159.440419,2.0
2019-01-03 00:00:00+00:00,58.042083,9.148248,43.88,69.55,57691.780937,7265.181910,46078.2500,65797.8950,942.652292,1684.822348,...,9.177808,42.017233,72.450550,25.67,23713.3950,3,1,2019,56.598649,3.0
2019-01-04 00:00:00+00:00,48.917083,7.091700,26.90,55.78,55457.005625,6982.070073,43833.4025,63279.6600,375.754687,664.261013,...,6.403313,35.307679,61.556285,28.88,19819.9925,4,1,2019,64.915423,4.0
2019-01-05 00:00:00+00:00,43.155417,14.022366,18.37,61.64,52843.881458,5805.721892,45028.6725,60174.8475,261.317813,465.700970,...,7.193736,31.537637,59.007893,43.27,27887.8975,5,1,2019,102.020127,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-27 00:00:00+00:00,59.074583,24.869555,-0.02,82.73,53093.031354,7597.601289,39422.9500,60498.1250,1932.559062,3370.291452,...,23.907486,-1.945013,89.281213,82.75,42395.0500,2,12,2023,106.523683,88.0
2023-12-28 00:00:00+00:00,16.670833,16.803733,-1.43,46.83,55965.022708,7561.289589,43187.7150,63863.7400,1556.631146,2712.088244,...,11.625205,-15.057713,28.597863,48.26,20329.8375,3,12,2023,102.341338,89.0
2023-12-29 00:00:00+00:00,6.605000,8.989161,-0.95,26.59,54341.245000,6885.347145,42627.0675,61626.5525,1425.278333,2496.200930,...,7.101128,-12.379893,19.215878,27.54,17873.2925,4,12,2023,91.073262,90.0


In [10]:
# --- 4) Features + Target zusammenführen ---

# Annahme: ri_daily hat Spalten 'profit' und 'cycles'
ri_profit.index.name = "date"
df_daily = daily_agg.merge(
    ri_profit[["profit", "cycles"]],
    left_index=True,
    right_index=True,
    how="inner"
)

In [11]:

# --- 5) Random Forest Test ---

feature_cols = [
    col for col in df_daily.columns
    if col not in ["profit", "cycles"]
]

X = df_daily[feature_cols].values
y = df_daily["profit"].values

n = len(df_daily)
split = int(n * 0.8)

X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)

print("Train R2:", r2_score(y_train, y_train_pred))
print("Val   R2:", r2_score(y_val, y_val_pred))
print("Val  MAE:", mean_absolute_error(y_val, y_val_pred))


Train R2: 0.9832380639328248
Val   R2: 0.4497770112837509
Val  MAE: 53.88957077772695


In [12]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

# --- 1) Features definieren ---
feature_cols = [
    col for col in df_daily.columns
    if col not in ["profit", "cycles"]
]

X = df_daily[feature_cols].values
y = df_daily["profit"].values

# --- 2) NaN-Fix: Imputer ---
imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)

# --- 3) Train/Test-Split ---
n = len(df_daily)
split = int(n * 0.8)

X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

# --- 4) Gradient Boosting Modell ---
model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

# --- 5) Vorhersagen ---
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)

# --- 6) Metriken ---
print("Train R2:", r2_score(y_train, y_train_pred))
print("Val   R2:", r2_score(y_val, y_val_pred))
print("Val  MAE:", mean_absolute_error(y_val, y_val_pred))



Train R2: 0.9661564075785795
Val   R2: 0.44913713777266584
Val  MAE: 52.10869185225531


In [13]:
fi = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
})

fi = fi.sort_values("importance", ascending=False)

print("\nFeature Importances (sortiert):")
print(fi)



Feature Importances (sortiert):
                                              feature  importance
1               epex_spot_60min_de_lu_eur_per_mwh_std    0.332512
28                                        epex_spread    0.295930
25                                      id_full_h_std    0.209181
27                                      id_full_h_max    0.064019
3               epex_spot_60min_de_lu_eur_per_mwh_max    0.033337
11            pv_forecast_d_minus_1_1000_de_lu_mw_max    0.006442
9             pv_forecast_d_minus_1_1000_de_lu_mw_std    0.006329
8            pv_forecast_d_minus_1_1000_de_lu_mw_mean    0.005786
14  wind_offshore_forecast_d_minus_1_1000_de_lu_mw...    0.004920
13  wind_offshore_forecast_d_minus_1_1000_de_lu_mw...    0.004353
20                                 residual_load_mean    0.004006
24                                     id_full_h_mean    0.003864
23                                  residual_load_max    0.003276
26                                      id_